# Lab 3: Contextual Bandit-Based News Article Recommendation

**`Course`:** Reinforcement Learning Fundamentals  

**`Student Name`:** Dipti Dhawade 

**`Roll Number`:** U20230146 

**`GitHub Branch`:** dipti_U20230146  

# Imports and Setup

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score

from rlcmab_sampler import sampler


# Load Datasets

In [2]:
# Load data
train_users = pd.read_csv("data/train_users.csv")
test_users = pd.read_csv("data/test_users.csv")

# Separate features and target
X_train_full = train_users.drop(columns=['label', 'user_id'])
y_train_full = train_users['label']
X_test = test_users.drop(columns=['user_id'])

## Data Preprocessing

In this section:
- Handle missing values
- Encode categorical features
- Prepare data for user classification

In [3]:
# Handle missing values in 'age' column
X_train_full['age'].fillna(X_train_full['age'].median(), inplace=True)
X_test['age'].fillna(X_test['age'].median(), inplace=True)

# Encode categorical features (handle unseen categories)
label_encoders = {}
categorical_cols = ['browser_version', 'region_code']

for col in categorical_cols:
    le = LabelEncoder()
    # Fit on training data
    X_train_full[col] = le.fit_transform(X_train_full[col].astype(str))
    
    # Handle unseen categories in test data
    test_values = X_test[col].astype(str)
    encoded_test = []
    
    for val in test_values:
        if val in le.classes_:
            encoded_test.append(le.transform([val])[0])
        else:
            # Assign unseen categories to -1 (or max_value + 1)
            encoded_test.append(-1)
    
    X_test[col] = encoded_test
    label_encoders[col] = le

# Convert boolean to int
X_train_full['subscriber'] = X_train_full['subscriber'].astype(int)
X_test['subscriber'] = X_test['subscriber'].astype(int)

# Encode target labels
le_target = LabelEncoder()
y_train_encoded = le_target.fit_transform(y_train_full)

# Split for validation
X_train, X_val, y_train, y_val = train_test_split(
    X_train_full, y_train_encoded, test_size=0.2, random_state=42, stratify=y_train_encoded
)

print(f"Training set shape: {X_train.shape}")
print(f"Validation set shape: {X_val.shape}")
print(f"Test set shape: {X_test.shape}")
print(f"\nTarget classes: {le_target.classes_}")

/var/folders/25/byfj08qs0vd_p5mct4_8s7fr0000gn/T/ipykernel_29541/1463850803.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  X_train_full['age'].fillna(X_train_full['age'].median(), inplace=True)
/var/folders/25/byfj08qs0vd_p5mct4_8s7fr0000gn/T/ipykernel_29541/1463850803.py:3: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we a

Training set shape: (1600, 31)
Validation set shape: (400, 31)
Test set shape: (2000, 31)

Target classes: ['user_1' 'user_2' 'user_3']


## User Classification

Train a classifier to predict the user category (`User1`, `User2`, `User3`),
which serves as the **context** for the contextual bandit.

In [4]:
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Initialize Gradient Boosting Classifier
gb_clf = GradientBoostingClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=5,
    min_samples_split=20,
    min_samples_leaf=10,
    subsample=0.8,
    random_state=42,
    verbose=1
)

# Train the model
print("Training Gradient Boosting Classifier...")
gb_clf.fit(X_train, y_train)

Training Gradient Boosting Classifier...
      Iter       Train Loss      OOB Improve   Remaining Time 
         1           0.9499           0.1341            1.85s
         2           0.8395           0.1091            1.84s
         3           0.7528           0.1026            1.83s
         4           0.6788           0.0741            1.81s
         5           0.6157           0.0561            1.78s
         6           0.5615           0.0420            1.76s
         7           0.5225           0.0689            1.74s
         8           0.4761           0.0203            1.72s
         9           0.4451           0.0359            1.70s
        10           0.4160           0.0268            1.68s
        20           0.2493          -0.0370            1.50s
        30           0.1712          -0.0295            1.31s
        40           0.1292          -0.0134            1.13s
        50           0.0981          -0.0283            0.94s
        60           0.0773 

GradientBoostingClassifier(max_depth=5, min_samples_leaf=10,
                           min_samples_split=20, random_state=42, subsample=0.8,
                           verbose=1)

In [5]:
# Validation predictions
y_val_pred = gb_clf.predict(X_val)
val_accuracy = accuracy_score(y_val, y_val_pred)

print(f"\n{'='*60}")
print(f"Validation Accuracy: {val_accuracy:.4f}")
print(f"{'='*60}")

# Detailed classification report
print("\nClassification Report (Validation Set):")
print(classification_report(y_val, y_val_pred, target_names=le_target.classes_))

# Confusion Matrix
print("\nConfusion Matrix (Validation Set):")
print(confusion_matrix(y_val, y_val_pred))

# Feature importance
feature_importance = pd.DataFrame({
    'feature': X_train.columns,
    'importance': gb_clf.feature_importances_
}).sort_values('importance', ascending=False)

print("\nTop 10 Most Important Features:")
print(feature_importance.head(10))

# Predict on test set
y_test_pred = gb_clf.predict(X_test)
test_users['predicted_user_category'] = le_target.inverse_transform(y_test_pred)

print(f"\nTest predictions completed!")
print(f"Predicted distribution:")
print(test_users['predicted_user_category'].value_counts())
print("\nFirst 5 test predictions:")
print(test_users[['user_id', 'predicted_user_category']].head())


Validation Accuracy: 0.9050

Classification Report (Validation Set):
              precision    recall  f1-score   support

      user_1       0.89      0.86      0.87       142
      user_2       0.99      0.88      0.93       142
      user_3       0.84      0.99      0.91       116

    accuracy                           0.91       400
   macro avg       0.91      0.91      0.91       400
weighted avg       0.91      0.91      0.91       400


Confusion Matrix (Validation Set):
[[122   1  19]
 [ 14 125   3]
 [  1   0 115]]

Top 10 Most Important Features:
                  feature  importance
4        session_duration    0.393106
29            region_code    0.333336
0                     age    0.055388
15  preferred_price_range    0.020434
5         content_variety    0.017631
13           time_on_site    0.014068
12        scroll_activity    0.011941
25        browser_version    0.010623
14      interaction_count    0.009682
20       churn_risk_score    0.009346

Test prediction

In [8]:
# ============================================================================
# CELL 3: TEST SET PREDICTIONS
# ============================================================================

import pandas as pd
import numpy as np

# Load test data
test_users_original = pd.read_csv("data/test_users.csv")

print("Making predictions on test set...")
print("="*60)

# Get predictions
y_test_pred = gb_clf.predict(X_test)
predicted_labels = le_target.inverse_transform(y_test_pred)

# Show prediction distribution
print("\nPredicted User Category Distribution:")
unique, counts = np.unique(predicted_labels, return_counts=True)
for label, count in zip(unique, counts):
    print(f"  {label}: {count} ({count/len(predicted_labels)*100:.2f}%)")

# Show sample predictions
print("\nSample Predictions (First 20):")
prediction_sample = pd.DataFrame({
    'user_id': test_users_original['user_id'][:20],
    'predicted_category': predicted_labels[:20]
})
print(prediction_sample.to_string(index=False))

# Save predictions to CSV
output_df = pd.DataFrame({
    'user_id': test_users_original['user_id'],
    'predicted_user_category': predicted_labels
})
output_df.to_csv('test_predictions.csv', index=False)
print("\n✓ Predictions saved to 'test_predictions.csv'")

print("\n" + "="*60)
print("Prediction Complete!")
print("="*60)

Making predictions on test set...

Predicted User Category Distribution:
  user_1: 674 (33.70%)
  user_2: 710 (35.50%)
  user_3: 616 (30.80%)

Sample Predictions (First 20):
user_id predicted_category
  U4058             user_2
  U1118             user_1
  U6555             user_1
  U9170             user_1
  U3348             user_1
  U2244             user_3
  U3022             user_3
  U5291             user_1
  U1945             user_3
  U6084             user_3
  U3714             user_2
  U0498             user_2
  U4264             user_3
  U8953             user_1
  U3285             user_3
  U0969             user_2
  U0173             user_3
  U9103             user_1
  U8708             user_3
  U3330             user_2

✓ Predictions saved to 'test_predictions.csv'

Prediction Complete!


# `Contextual Bandit`

## Reward Sampler Initialization

The sampler is initialized using the student's roll number `i`.
Rewards are obtained using `sampler.sample(j)`.


## Arm Mapping

| Arm Index (j) | News Category | User Context |
|--------------|---------------|--------------|
| 0–3          | Entertainment, Education, Tech, Crime | User1 |
| 4–7          | Entertainment, Education, Tech, Crime | User2 |
| 8–11         | Entertainment, Education, Tech, Crime | User3 |

## Epsilon-Greedy Strategy

This section implements the epsilon-greedy contextual bandit algorithm.


In [7]:
"""5.3 Contextual Bandit Algorithms (45 Points)
You must implement three distinct strategies. For each strategy, treat the User Category as the
context and News Category as the Arm.
5.3.1 Epsilon-Greedy (15 Points)
• Train a separate model for each of the 3 user contexts.
• Compute the Expected Reward Distribution for each news category across all contexts.
• Hyperparameter Tuning: Experiment with multiple values of ϵ. Compare the expected
payoffs for different ϵ values."""



'5.3 Contextual Bandit Algorithms (45 Points)\nYou must implement three distinct strategies. For each strategy, treat the User Category as the\ncontext and News Category as the Arm.\n5.3.1 Epsilon-Greedy (15 Points)\n• Train a separate model for each of the 3 user contexts.\n• Compute the Expected Reward Distribution for each news category across all contexts.\n• Hyperparameter Tuning: Experiment with multiple values of ϵ. Compare the expected\npayoffs for different ϵ values.'

## Upper Confidence Bound (UCB)

This section implements the UCB strategy for contextual bandits.

## SoftMax Strategy

This section implements the SoftMax strategy with temperature $ \tau = 1$.


## Reinforcement Learning Simulation

We simulate the bandit algorithms for $T = 10,000$ steps and record rewards.

P.S.: Change $T$ value as and if required.


## Results and Analysis

This section presents:
- Average Reward vs Time
- Hyperparameter comparisons
- Observations and discussion


## Final Observations

- Comparison of Epsilon-Greedy, UCB, and SoftMax
- Effect of hyperparameters
- Strengths and limitations of each approach
